In [116]:
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt
import os

In [117]:
df = pd.read_json("combined_dataset.json")

In [118]:
df.head()

,url,type,course_name,organization,instructor,rating,nu_reviews,description,skills,level,...,has_teammate,has_peer,has_discussion,has_video,has_no_enrol,enrollments,has_rating,subject,has_subject,provider
0,https://www.coursera.org/learn/serverless-comp...,course,AWS Lambda إنشاء صورة مصغرة بإستخدام السيرفرل...,[Coursera Project Network],Omar Fathy,No rating,0,Description: هذا المشروع التفاعلي -إنشاء صورة ...,"[AWS Identity And Access Management (IAM), Clo...",Intermediate,...,1,1,1,1,0,NaN,1,None,0,coursera
1,https://www.coursera.org/learn/assist-public-s...,course,Assisting Public Sector Decision Makers With ...,[University of Michigan],Christopher Brooks,4.8,22,Description: Develop data analysis skills that...,"[Simulations, Statistical Analysis, Predictive...",Intermediate,...,1,1,1,1,0,NaN,1,None,0,coursera
2,https://www.coursera.org/learn/advanced-strate...,course,Advanced Strategies for Sustainable Business,[University of Colorado Boulder],Joel Hartter,No rating,0,Description: This course focuses on integratin...,"[Circular Economy, Sustainable Business, Stake...",Beginner,...,1,1,1,1,0,NaN,1,None,0,coursera
3,https://www.coursera.org/learn/applying-machin...,course,Applying Machine Learning to Your Data with G...,[Google Cloud],Google Cloud Training,No rating,0,"Description: Dans ce cours, nous définirons ce...",[NaN],Beginner,...,1,1,1,1,0,NaN,1,None,0,coursera
4,https://www.coursera.org/projects/automate-blo...,project,Automate Blog Advertisements with Zapier,[Coursera Project Network],Carmen Rojas,No rating,0,Description: Zapier is the industry leader in ...,"[Advertising, Social Media, Blogging, Marketing]",Intermediate,...,0,0,0,0,0,NaN,1,None,0,coursera


In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13793 entries, 0 to 13792
Data columns (total 40 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   url                13793 non-null  object 
 1   type               13793 non-null  object 
 2   course_name        13793 non-null  object 
 3   organization       13793 non-null  object 
 4   instructor         13793 non-null  object 
 5   rating             13174 non-null  object 
 6   nu_reviews         13174 non-null  object 
 7   description        13793 non-null  object 
 8   skills             13793 non-null  object 
 9   level              13793 non-null  object 
 10  Duration           12560 non-null  float64
 11  reviews            13156 non-null  object 
 12  total_assignment   11011 non-null  float64
 13  total_app          11011 non-null  float64
 14  total_programming  11011 non-null  float64
 15  total_reading      11011 non-null  float64
 16  total_plugin       110

In [120]:
df.isnull().sum()

url                      0
type                     0
course_name              0
organization             0
instructor               0
rating                 619
nu_reviews             619
description              0
skills                   0
level                    0
Duration              1233
reviews                637
total_assignment      2782
total_app             2782
total_programming     2782
total_reading         2782
total_plugin          2782
total_ungraded        2782
total_quiz            2782
total_teammate        2782
total_peer            2782
total_discussion      2782
total_video           2782
has_assignment           0
has_app                  0
has_programming          0
has_reading              0
has_plugin               0
has_ungraded             0
has_quiz                 0
has_teammate             0
has_peer                 0
has_discussion           0
has_video                0
has_no_enrol             0
enrollments          13174
has_rating               0
s

In [121]:
new_df = df[
    [
        "course_name",
        "description",
        "skills",
        "organization",
        "provider",
        "level",
        "rating",
        "Duration",
        "reviews",
        "instructor",
        "url"
    ]
].copy()

In [122]:
new_df["rating"] = new_df["rating"].fillna(0)

new_df["reviews"] = new_df["reviews"].fillna(0)

new_df["Duration"] = new_df["Duration"].fillna(
    new_df["Duration"].median()
)

In [123]:
new_df.isnull().sum()

course_name     0
description     0
skills          0
organization    0
provider        0
level           0
rating          0
Duration        0
reviews         0
instructor      0
url             0
dtype: int64

In [124]:
print(type(new_df.loc[0, "skills"]))
print(type(new_df.loc[0, "description"]))
print(type(new_df.loc[0, "course_name"]))

<class 'list'>
<class 'str'>
<class 'str'>


In [125]:
new_df["skills"] = new_df["skills"].apply(
    lambda x: " ".join(x) if isinstance(x, list) else str(x)
)

In [126]:
text_columns = [
    "course_name",
    "description",
    "skills",
    "organization",
    "provider",
    "level",
    "instructor"
]

for col in text_columns:
    new_df[col] = new_df[col].astype(str)

In [127]:
new_df["tags"] = (
    new_df["course_name"] + " " +
    new_df["description"] + " " +
    new_df["skills"] + " " +
    new_df["organization"] + " " +
    new_df["provider"] + " " +
    new_df["level"] + " " +
    new_df["instructor"]
)

In [128]:
new_df["tags"] = new_df["tags"].str.lower()

In [129]:
print(new_df["tags"].head())

0     aws lambda إنشاء صورة مصغرة بإستخدام السيرفرل...
1     assisting public sector decision makers with ...
2     advanced strategies for sustainable business ...
3     applying machine learning to your data with g...
4     automate blog advertisements with zapier  des...
Name: tags, dtype: object


In [143]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [144]:
def remove_stopwords(text):
    return [word for word in text if word.lower() not in stop_words]

new_df["tags"] = new_df["tags"].apply(remove_stopwords)

In [145]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y = []
    for i in text:
        y.append(ps.stem(i))
    return y

new_df["tags"] = new_df["tags"].apply(stem)

In [146]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))

In [147]:
new_df["tags"].head()

0        w       l   b       إ   ن   ش   ا   ء     ...
1        n   g       p   u   b   l   c       e   c ...
2        v   n   c   e       r   e   g   e       f ...
3        p   p   l   n   g       c   h   n   e     ...
4        u   e       b   l   g       v   e   r   e ...
Name: tags, dtype: object

In [148]:
new_df["tags"].str.len().head()

0    6761
1    9321
2    5869
3    4505
4    3889
Name: tags, dtype: int64

In [149]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)

vectors = tfidf.fit_transform(new_df["tags"])

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [150]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)

In [151]:
import pickle
import os

os.makedirs("models", exist_ok=True)

pickle.dump(similarity, open("models/similarity.pkl", "wb"))

pickle.dump(df, open("models/courses.pkl", "wb"))